# Microstructure and a persistence baseline

This notebook follows the concepts in the supplied [Optiver introduction](https://www.kaggle.com/code/jiashenliu/introduction-to-financial-concepts-and-data), using newly written explanations and code. The illustrative numbers below are not competition observations.

A bid is a displayed buying price; an ask is a displayed selling price. The gap is the spread. Sizes describe how many shares are displayed at those prices. Book interest and executed trades describe different aspects of activity.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Open this notebook from the repository or notebooks directory"


## WAP uses the opposite side's size

$$WAP=\frac{bid\times asksize+ask\times bidsize}{bidsize+asksize}$$

When bid size is larger, this formula puts more weight on the ask price. Do not substitute ordinary same-side volume weights. WAP is a book-based proxy, not an actual execution price.


In [ ]:
bid, ask, bid_size, ask_size = 99.9, 100.1, 30, 10
wap = (bid * ask_size + ask * bid_size) / (bid_size + ask_size)
print(f"Illustrative WAP: {wap:.4f}")


## Returns and realized volatility

$$r_t=\log(P_t/P_{t-1}),\quad RV=\sqrt{\sum_t r_t^2}$$

Positive and negative changes both increase RV because they are squared. The competition measure is not annualized. The first observation in a window has no previous observation and therefore no return.


In [ ]:
from optiver_volatility.features import realized_volatility
prices = pd.Series([100.0, 100.1, 99.9, 100.2])
returns = np.log(prices).diff()
print(pd.DataFrame({"illustrative_price": prices, "log_return": returns}))
print("Illustrative observed-window RV:", realized_volatility(returns))


## Persistence turns a measurement into a forecast

Observed-window RV is a feature. Predicting next-window RV by that value is a persistence forecast. It exploits the tendency for volatility to cluster, but can miss changes in market conditions.

The source notebook's example has observed RV about 0.004499 and future target about 0.004136. Its all-training-row persistence RMSPE is 0.341. That result is a useful reference benchmark and is not an OOF result from this project.


## RMSPE measures relative prediction error

$$RMSPE=\sqrt{\operatorname{mean}(((y-\hat y)/y)^2)}$$

Lower is better. An error of 0.001 is much more serious when true volatility is 0.002 than when it is 0.02. This differs from absolute RMSE. R² answers a different question and is not the primary competition metric.


In [ ]:
from optiver_volatility.metrics import rmspe
illustrative_target = np.array([0.002, 0.02])
illustrative_prediction = np.array([0.003, 0.021])
print("Relative errors:", (illustrative_prediction - illustrative_target) / illustrative_target)
print("Illustrative RMSPE:", rmspe(illustrative_target, illustrative_prediction))


## Before moving on

Check that you can explain why the observed RV is not the label, why up/down moves both contribute to volatility, and why returns cannot be differenced across two `time_id` values. The production functions enforce these boundaries.
